# Policy Document Search AI

Policy Document Search AI aims to build a RAG system that can read and understand insurance policy PDFs.

It extracts and indexes policy texts, combines semantic search (embeddings) 

## Problem Statement
Build a Retrieval-Augmented Generation (RAG) system that answers questions about insurance policies from long PDF documents. The system must extract text, index it for efficient retrieval, and generate grounded answers with citations (policy name + page) so users can quickly find eligibility, coverage, exclusions, and termination details.

## Why LlamaIndex
- Provides **simple connectors** for PDFs and a clean **node/chunk** abstraction.
- Pluggable **embeddings**, **vector stores (Chroma)**, and **LLMs** with minimal glue code.
- Built-in **retrievers** and **node_postprocessors** (e.g., cross-encoder re-rank) to improve precision.
- **Response synthesizer** controls answer style and citations, keeping outputs grounded.
- Scales from a single notebook to app frameworks (e.g., Streamlit) with the same APIs.


## Ingestions

In [1]:
from llama_index.core import SimpleDirectoryReader

reader = SimpleDirectoryReader('documents')
documents = reader.load_data(num_workers=2)

In [2]:
import pprint

pprint.pprint(documents[0])

Document(id_='e11d92f3-b65d-4dba-8477-64198f0cd403', embedding=None, metadata={'page_label': '1', 'file_name': 'HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf', 'file_path': '/Users/rishabhsaha/repo/Policy-Document-Search/documents/HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf', 'file_type': 'application/pdf', 'file_size': 1303156, 'creation_date': '2025-09-12', 'last_modified_date': '2025-09-25'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text=' \n \n            Part A \n<<Date>> \n<<Policyholder’s Name>>  \n<<Policyholder’s Address>> \n<<Policyholder’s Contact Number>> \n \nDear <<Policyholder’s Name>>,  \n \nS

## Extract

In [3]:
documents[0].__dict__

{'id_': 'e11d92f3-b65d-4dba-8477-64198f0cd403',
 'embedding': None,
 'metadata': {'page_label': '1',
  'file_name': 'HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf',
  'file_path': '/Users/rishabhsaha/repo/Policy-Document-Search/documents/HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf',
  'file_type': 'application/pdf',
  'file_size': 1303156,
  'creation_date': '2025-09-12',
  'last_modified_date': '2025-09-25'},
 'excluded_embed_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'excluded_llm_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'relationships': {},
 'metadata_template': '{key}: {value}',
 'metadata_separator': '\n',
 'text_resource': MediaResource(embeddings=None, data=None, text=' \n \n            Part A \n<<Date>> \n<<Policyholder’s Name>>  \n<<Policyholder’s Address>> \n<<Policyholder’s 

In [4]:
import openai
import dotenv
import os

dotenv.load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

In [5]:
from llama_index.llms.openai import OpenAI

llm_transformations = OpenAI(model="gpt-3.5-turbo")

In [6]:
from llama_index.core.extractors import (TitleExtractor, QuestionsAnsweredExtractor)
from llama_index.core.ingestion import IngestionPipeline, IngestionCache
from llama_index.core.node_parser import SentenceSplitter


title_extractor = TitleExtractor(llm=llm_transformations)
questions_answered_extractor = QuestionsAnsweredExtractor(llm=llm_transformations, questions=3)

text_splitter = SentenceSplitter(
    separator=" ", chunk_size=1024, chunk_overlap=128
)



ingestion_pipeline = IngestionPipeline(
    transformations=[
        text_splitter,
        title_extractor,
        questions_answered_extractor
    ],
    cache=IngestionCache(),
)

In [7]:
chunks = ingestion_pipeline.run(
    documents = documents,
    in_place = True,
    show_progress = True
)

/Users/rishabhsaha/repo/Policy-Document-Search/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 241/241 [00:54<00:00,  4.43it/s]


In [8]:
len(chunks)


241

In [9]:
pprint.pprint(chunks[0])


TextNode(id_='239de4d3-0035-420d-aa05-a92a1a100960', embedding=None, metadata={'page_label': '1', 'file_name': 'HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf', 'file_path': '/Users/rishabhsaha/repo/Policy-Document-Search/documents/HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf', 'file_type': 'application/pdf', 'file_size': 1303156, 'creation_date': '2025-09-12', 'last_modified_date': '2025-09-25', 'document_title': 'Policy Issuance and Contact Information for Candidates', 'questions_this_excerpt_can_answer': '1. What are the specific steps and conditions for cancelling the policy during the Free-Look Period, including the timeframe for returning the policy and the process for receiving a refund?\n2. How can the policyholder contact HDFC Life Insurance Company Limited for any queries or grievances, and what are the details of the Certified Financial Consultant who advised them on the policy?\n3. What information is included in the Policy document that is enclo

In [10]:
from llama_index.core.schema import MetadataMode

print(chunks[0].get_content(metadata_mode=MetadataMode.LLM))

[Excerpt from document]
page_label: 1
file_path: /Users/rishabhsaha/repo/Policy-Document-Search/documents/HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
document_title: Policy Issuance and Contact Information for Candidates
questions_this_excerpt_can_answer: 1. What are the specific steps and conditions for cancelling the policy during the Free-Look Period, including the timeframe for returning the policy and the process for receiving a refund?
2. How can the policyholder contact HDFC Life Insurance Company Limited for any queries or grievances, and what are the details of the Certified Financial Consultant who advised them on the policy?
3. What information is included in the Policy document that is enclosed with this communication, and why is it important for the policyholder to preserve it safely and inform their nominees about it?
Excerpt:
-----
Part A 
<<Date>> 
<<Policyholder’s Name>>  
<<Policyholder’s Address>> 
<<Policyholder’s Contact Number>> 
 
Dear <<Policyhol

## Index

In [11]:
from llama_index.embeddings.openai import OpenAIEmbedding

embedding = OpenAIEmbedding(model="text-embedding-3-small")

In [12]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex(chunks, embedding=embedding)

In [13]:
llm_query = OpenAI(model="gpt-3.5-turbo")
query_engine = index.as_query_engine(llm=llm_query)
response = query_engine.query("What is this document about?")
print(response)

The document is about providing an overview of the Insurance Laws (Amendment) Act, 2015, including a simplified version for general information. It also includes a disclaimer advising policy holders to refer to the complete and accurate details in the actual Insurance Laws (Amendment) Act, 2015 dated 23.03.2015.


In [14]:
response.__dict__

{'response': 'The document is about providing an overview of the Insurance Laws (Amendment) Act, 2015, including a simplified version for general information. It also includes a disclaimer advising policy holders to refer to the complete and accurate details in the actual Insurance Laws (Amendment) Act, 2015 dated 23.03.2015.',
 'source_nodes': [NodeWithScore(node=TextNode(id_='6d560147-82b6-4ea8-b52d-6d1b30aa99ca', embedding=None, metadata={'page_label': '30', 'file_name': 'HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf', 'file_path': '/Users/rishabhsaha/repo/Policy-Document-Search/documents/HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf', 'file_type': 'application/pdf', 'file_size': 1303156, 'creation_date': '2025-09-12', 'last_modified_date': '2025-09-25', 'document_title': 'Overview of the Insurance Laws (Amendment) Act, 2015', 'questions_this_excerpt_can_answer': '1. What is the disclaimer provided regarding the overview of the Insurance Laws (Amendment) 

## Vector Store

In [15]:
%pip install -Uq chromadb
%pip install -Uq llama-index-vector-stores-chroma

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [16]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext, VectorStoreIndex

db = chromadb.PersistentClient(path="./chroma_db")

chroma_collection = db.get_or_create_collection(name="policyGPT")

vector_store = ChromaVectorStore(chroma_collection)
storage_context= StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex(chunks, storage_context=storage_context, embedding=embedding)


## Making of the Response Engines

There is more to querying that intially meets the eye. Querying consists of 3 distinct stages:

1. Retrieval: Fetch relevant documents from the vector store based on the query.
2. Post-Processing: Refine and rerank the retrieved documents to ensure the most relevant ones are prioritized.
3. Synthesis: Generate a coherent and contextually relevant response using the refined documents.

Trying out TimeWeightedPostprocessor

In [17]:
from llama_index.core.postprocessor import TimeWeightedPostprocessor

llm = OpenAI(model="gpt-3.5-turbo")
query_engine = index.as_query_engine(
    llm=llm, 
    node_postprocessors=[
        TimeWeightedPostprocessor(
            time_decay=0.5, time_access_refresh=False, top_k=1
        )
    ],
    streaming=True
)


In [18]:
def response_stream(query):
    streaming_response = query_engine.query(query)
    streaming_response.print_response_stream()
    # return streaming_response

In [19]:
response_stream("what is criteria for HDFC group insurance?")

The criteria for joining the HDFC Life Group Poorna Suraksha scheme includes minimum and maximum age limits for different plan options, minimum number of members required to join the scheme, and the various modes/frequencies of premium payment available for policyholders.

### Similarity and Reranking Postprocessors

This section demonstrates how to enhance retrieval quality using `SimilarityPostprocessor` for filtering by similarity and `SentenceTransformerRerank` for cross-encoder reranking. The resulting query engine returns top-ranked, contextually relevant document chunks.

In [20]:
from llama_index.core.postprocessor import SimilarityPostprocessor, SentenceTransformerRerank

similarity_postprocessor = SimilarityPostprocessor(similarity_cutoff=0.15)
cross_encoder_rerank = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n=5
)

rerank_query_engine = index.as_retriever(
    llm=llm,                 
    similarity_top_k=10,
    node_postprocessors=[similarity_postprocessor, cross_encoder_rerank],
    streaming=True
)


In [21]:
from time import time

def reranked_response_stream(query):
    now = time()
    rerank_response = rerank_query_engine.retrieve(query)
    # rerank_response.print_response_stream()
    # print(rerank_response)
    print(f"Elapsed: {round(time() - now, 2)}s")
    return rerank_response

In [22]:
response = reranked_response_stream("what is criteria for HDFC group insurance?")
# print(response.get_formatted_sources())  # check sources

# convert response (list of NodeWithScore) into a dataframe
import pandas as pd
df = pd.DataFrame({
	'documents': [node.text for node in response],
	'similarity': [node.score for node in response],
	'node_id': [node.node.node_id for node in response],
	'title': [node.node.metadata.get('document_title', 'N/A') for node in response],
	'questions_answered': [node.node.metadata.get('questions_this_excerpt_can_answer', 'N/A') for node in response]
})

# df
# response[0]

Elapsed: 0.26s


## Chat Engines - Response Synthesis


In [23]:
from llama_index.core import PromptTemplate

chat_llm = OpenAI(model="gpt-3.5-turbo")

QA_PROMPT = PromptTemplate(
    """You are a helpful assistant in the insurance domain who can effectively answer user queries about insurance policies and documents.
    You have a question asked by the user in {query_str} and you have some search results from a corpus of insurance documents in the dataframe '{top_3_RAG}'. These search results are essentially one page of an insurance document that may be relevant to the user query.

    The column 'document' inside this dataframe contains the actual text from the policy document.
    The column 'file_name' contains the policy name.
    The column 'page_label' contains the source page. 
    The text inside the document may also contain tables in the format of a list of lists where each of the nested lists indicates a row.

    Use the documents in '{top_3_RAG}' to answer the query '{query_str}'. Frame an informative answer and also, use the dataframe to return the relevant policy names and page numbers as citations.

    Follow the guidelines below when performing the task.
    1. Try to provide relevant/accurate numbers if available.
    2. You don’t have to necessarily use all the information in the dataframe. Only choose information that is relevant.
    3. If the document text has tables with relevant information, please reformat the table and return the final information in a tabular in format.
    3. Use the Metadatas columns in the dataframe to retrieve and cite the policy name(s) and page numbers(s) as citation.
    4. If you can't provide the complete answer, please also provide any information that will help the user to search specific sections in the relevant cited documents.
    5. You are a customer facing assistant, so do not provide any information on internal workings, just answer the query directly.

    The generated response should answer the query directly addressing the user and avoiding additional information. If you think that the query is not relevant to the document, reply that the query is irrelevant. Provide the final response as a well-formatted and easily readable text along with the citation. Provide your complete response first with all information, and then provide the citations.
    """
)

In [24]:
def generate_response(retrieved_nodes, query_str, qa_prompt, llm):
    top_3_RAG = pd.DataFrame({
        'document': [node.text for node in retrieved_nodes],
        'similarity': [node.score for node in retrieved_nodes],
        'node_id': [node.node.node_id for node in retrieved_nodes],
        'title': [node.node.metadata.get('file_name', 'N/A') for node in retrieved_nodes],
        'page_label': [node.node.metadata.get('page_label', 'N/A') for node in retrieved_nodes],
        'questions_answered': [node.node.metadata.get('questions_this_excerpt_can_answer', 'N/A') for node in retrieved_nodes]
    })
    fmt_qa_prompt = qa_prompt.format(
        top_3_RAG=top_3_RAG, query_str=query_str
    )
    response = llm.complete(fmt_qa_prompt)
    # summarizer = TreeSummarize(verbose=True, summary_template=qa_prompt)
    # response = summarizer.summarize(fmt_qa_prompt)

    return str(response), fmt_qa_prompt

In [25]:
query_str = "what are HDFC Life Sanchay Plus Life Long Income Option ?"
retrieved_nodes=rerank_query_engine.retrieve(query_str)

response, fmt_qa_response = generate_response(
    retrieved_nodes, query_str, qa_prompt=QA_PROMPT, llm=chat_llm
    
)

print(f"*****Response******:\n{response}\n\n")

*****Response******:
The HDFC Life Sanchay Plus Life Long Income Option is a feature offered in the HDFC Life Sanchay Plus insurance policy. This option provides a guaranteed income for life, ensuring financial security in the long term. For more specific details and options available under this feature, please refer to the HDFC Life Sanchay Plus policy document.

Citation:
- Policy Name: HDFC Life Sanchay Plus
- Relevant Page Number: 3, 8, 25, 27, 17, 12, 11, 1, 15, 14


